# ScamSense — Notebook 04: LangGraph Agentic Prediction Pipeline

This notebook implements the ScamSense **agentic pipeline** using **LangGraph**.
Three specialised agents communicate through a shared state graph, with conditional
routing that decides whether to run the full scam analysis or exit early for safe messages.

| Agent | Role | Method |
|---|---|---|
| **Detection Agent** | Classify Scam / Ham + detect language | Fine-tuned XLM-RoBERTa |
| **Explanation Agent** | SHAP token attribution + **RAG over SPF knowledge base** | SHAP + FAISS |
| **Risk Agent** | Scam type + risk score + SPF 2025 statistics | Rule-based SPF taxonomy |

**Why LangGraph?**  
LangGraph enforces a proper agent architecture: each agent is a node, state is passed
explicitly between nodes, and conditional edges allow the graph to branch — for example,
bypassing the Explanation and Risk agents entirely when a message is clearly safe.

```
User Message
      │
      ▼
┌─────────────────┐
│ Detection Agent │  ← XLM-RoBERTa
└────────┬────────┘
         │
    Ham (≥80%)?
    ┌────┴─────┐
   Yes         No
    │          │
    ▼          ▼
┌────────┐  ┌──────────────────┐
│  Safe  │  │ Explanation Agent│  ← SHAP + FAISS RAG (SPF corpus)
│  Exit  │  └────────┬─────────┘
└────────┘           │
                     ▼
              ┌─────────────┐
              │  Risk Agent │  ← SPF 2025 taxonomy
              └─────────────┘
```



## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# Run this cell first, then restart the kernel when prompted

packages = [                              # pip packages needed by this notebook
    "langgraph",                          # state-machine framework for the 3 agents
    "langdetect",                         # language detection fallback
    "shap",                               # SHAP token-importance explanations
    "sentence-transformers==3.0.1",       # multilingual sentence embeddings for RAG
    "faiss-cpu",                          # vector similarity search for RAG retrieval
    "numpy<2",                            # pin numpy for faiss/shap compatibility
    "pandas",                             # dataframes for displaying results
    "tabulate",                           # pretty-printing tables
]

import subprocess, sys                    # subprocess to run pip; sys for the python executable path
for pkg in packages:                       # install each package one at a time
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])  # quiet pip install
    print(f"  ✔ {pkg}")                    # confirm this package installed

#All packages installed — restart kernel now
import os; os.kill(os.getpid(), 9)         # force-restart the kernel so new packages load


  ✔ langgraph
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.1 MB/s eta 0:00:00
  ✔ langdetect
  ✔ shap
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.9 MB/s eta 0:00:00
  ✔ sentence-transformers==3.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 91.1 MB/s eta 0:00:00
  ✔ faiss-cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 w

  ✔ numpy<2
  ✔ pandas


## Cell 2 — Shared Pipeline Module (`scamsense_pipeline.py`)

This is the **only** copy of the classification/explanation/risk-scoring logic — it is not duplicated in this notebook. Both `04_langgraph_rag.ipynb` and `05_fastapi_backend.ipynb` import the exact same file.

**Setup (do this once):** upload `scamsense_pipeline.py` as a Kaggle Dataset (or Utility Script) and attach it to this notebook as input. The loader cell below checks a few candidate folders in order — your current working directory, `/kaggle/input/datasets/bhoovika/scamsense-utils` (a Kaggle dataset path), and `/kaggle/working` — and uses whichever one contains the file. If you named your dataset differently, or you're running locally/Colab with `scamsense_pipeline.py` kept next to this notebook, just make sure one of those paths matches (or add your own to `_CANDIDATE_DIRS`).


In [1]:
import sys                                        # to modify the module search path
from pathlib import Path                          # for filesystem path handling

# scamsense_pipeline.py lives in exactly one place; every notebook just points
# at it. Adjust the Kaggle dataset slug below if you named it differently.
_CANDIDATE_DIRS = [                                           # possible locations of the shared module
    Path.cwd(),                                               # same folder (local / Colab)
    Path("/kaggle/input/datasets/bhoovika/scamsense-utils"),  # Kaggle dataset/utility script
    Path("/kaggle/working"),                                  # already copied into working dir
]

for _dir in _CANDIDATE_DIRS:                       # check each candidate folder in order
    if (_dir / "scamsense_pipeline.py").exists():  # does the shared module live here?
        sys.path.insert(0, str(_dir))              # make it importable
        break                                       # stop at the first match found
else:                                               # runs only if no candidate matched
    raise FileNotFoundError(                        # fail loudly with guidance
        "scamsense_pipeline.py not found in any of: "
        + ", ".join(str(d) for d in _CANDIDATE_DIRS)
        + ". Upload it as a Kaggle dataset/utility script (and add its path to "
        "_CANDIDATE_DIRS above), or place it next to this notebook."
    )

print(f"scamsense_pipeline.py loaded from: {_dir}")  # confirm which path was used


scamsense_pipeline.py loaded from: /kaggle/input/datasets/bhoovika/scamsense-utils


## Cell 3 — Imports, Secrets & Pipeline Init

In [2]:
import json                                    # (kept for downstream JSON handling)
from pathlib import Path                       # filesystem paths

import pandas as pd                            # dataframes for tables
from IPython.display import display            # render dataframes nicely in the notebook

from scamsense_pipeline import (               # import shared pipeline functions/data
    init, classify, explain, get_risk, run_pipeline,
    classify_scam_type, detect_language,
    SPF_TAXONOMY, SPF_CORPUS,
)

# Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient   # Kaggle's secrets API
    secrets  = UserSecretsClient()                  # secrets client instance
    HF_TOKEN = secrets.get_secret("HF_TOKEN")        # HuggingFace token, if configured
except Exception:
    HF_TOKEN = None                                  # fall back to no token (public model)

BASE_DIR = Path("/kaggle/working/ScamSense")     # root working directory for this project
RAG_DIR  = BASE_DIR / "rag"                       # subfolder for RAG index/corpus files
RAG_DIR.mkdir(parents=True, exist_ok=True)        # create it if it doesn't exist yet

print(f"Base dir: {BASE_DIR}")                    # confirm working directory

# Loads the classifier, embedder, SHAP explainer, and builds/caches the FAISS
# index from the module's built-in SPF corpus (first run only).
init(hf_token=HF_TOKEN, rag_dir=RAG_DIR)          # one-time setup of the shared pipeline

# Sanity check
test = classify("You won $10,000! Claim now.")    # run a quick test classification
print(f"\nSanity check -> label={test['label']} | confidence={test['confidence']:.1%}")  # show result


Base dir: /kaggle/working/ScamSense
Device: cuda
Loading tokenizer/model from Bhoovika/scamsense-xlmroberta ...


tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded
SPF corpus loaded from built-in default (40 chunks)
Building FAISS index from SPF corpus ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index built (40 vectors, dim=384)
   Cached → /kaggle/working/ScamSense/rag/spf_faiss.index
SHAP explainer ready
scamsense_pipeline ready

Sanity check -> label=scam | confidence=100.0%


## Cell 4 — SPF 2025 Risk Taxonomy

The taxonomy now lives in `scamsense_pipeline.SPF_TAXONOMY` (imported above) instead of being redefined here. Quick sanity check of the keyword matcher:

In [3]:
def show_taxonomy_result(text):                    # pretty-print a taxonomy match for one message
    scam_type, risk_score, risk_tier, advice = classify_scam_type(text)  # keyword-match the text
    info = SPF_TAXONOMY[scam_type]                  # look up the full taxonomy entry

    print(f"INPUT: {text}")                          # show the input message
    summary = pd.DataFrame([{                         # build a one-row summary table
        "Detected Type": info["spf_name"],             # human-readable scam name
        "Taxonomy Key": scam_type,                      # internal taxonomy key
        "Risk Tier": risk_tier,                          # CRITICAL / HIGH / MEDIUM / LOW
        "Risk Score": f"{risk_score}/100",               # numeric score out of 100
        "SPF Cases": f"{info['2025_cases']:,}" if info["2025_cases"] is not None else "N/A",      # 2025 case count
        "Total Losses": f"${info['2025_losses']}M" if info["2025_losses"] is not None else "N/A", # 2025 losses
        "Avg Loss": f"${info['avg_loss']:,}" if info["avg_loss"] is not None else "N/A",           # average loss/victim
    }])
    display(summary.style.hide(axis="index"))         # show table without the index column
    print("\nADVICE: ")                                # section header
    print(advice)                                       # print the SPF advisory text
    print()                                             # blank line for spacing

show_taxonomy_result("guaranteed 300% returns on crypto investment!")               # test: investment scam
show_taxonomy_result("Hi Mum, it's me, new number, please transfer $500 urgently")  # test: fake-friend scam

print(f"SPF taxonomy ready - {len(SPF_TAXONOMY)} categories aligned with NB01 synthetic labels + unknown fallback")  # confirm taxonomy size


INPUT: guaranteed 300% returns on crypto investment!


Detected Type,Taxonomy Key,Risk Tier,Risk Score,SPF Cases,Total Losses,Avg Loss
Investment Scam,investment,CRITICAL,95/100,"5,462",$336.2M,"$61,559"



ADVICE: 
Investment scams caused the HIGHEST losses in Singapore in 2025 — $336.2 million (SPF 2025). Never transfer money to strangers for 'investments'. Verify at MAS (mas.gov.sg/investor-alert-list). Report to SPF at 1800-255-0000.

INPUT: Hi Mum, it's me, new number, please transfer $500 urgently


Detected Type,Taxonomy Key,Risk Tier,Risk Score,SPF Cases,Total Losses,Avg Loss
Fake Friend Call Scam,fake_friend,MEDIUM,62/100,"1,551",$4.7M,"$3,056"



ADVICE: 
Fake friend call scams fell sharply in 2025 but still cost $4.7M across 1,551 cases (SPF 2025). Always verify a claimed new number by calling the person's old number or contacting them another way before sending money.

SPF taxonomy ready - 13 categories aligned with NB01 synthetic labels + unknown fallback


## Cell 5 — SPF Advisory Corpus + FAISS Index

The 40-passage SPF advisory corpus and its FAISS index are owned by `scamsense_pipeline` now (built and cached inside `init()` above). `SPF_CORPUS` is imported here for reference/display, and this cell also runs a quick retrieval test using `retrieve_spf_passages()` to confirm the index is working.


In [4]:
print(f"SPF corpus loaded: {len(SPF_CORPUS)} advisory chunks")                       # show corpus size
print(f"Scam types covered: {sorted(set(c['scam_type'] for c in SPF_CORPUS))}")       # list unique scam types in corpus

# Quick retrieval test (via the shared explain() helper, using its retrieval step directly)
from scamsense_pipeline import retrieve_spf_passages          # import the RAG retrieval function directly
results = retrieve_spf_passages("phishing bank account OTP", top_k=3)  # retrieve top-3 matches for a sample query
print("\nRetrieval test (phishing query):")                    # section header
for r in results:                                                # loop over each retrieved passage
    print(f"  [{r['score']:.3f}] {r['id']} - {r['topic']}")      # print similarity score, id, and topic


SPF corpus loaded: 0 advisory chunks
Scam types covered: []

Retrieval test (phishing query):
  [0.721] spf_019 - Phishing contact methods
  [0.686] spf_018 - Phishing tactics — card details and OTP theft
  [0.537] spf_031 - Bank impersonation as a phishing sub-pattern


## Cell 6 — Agent State Schema

SHAP explanation and language detection now live in `scamsense_pipeline` (`explain()` and `detect_language()`). This cell just defines the LangGraph state schema shared by the three agents below.

In [5]:
from typing import TypedDict, Optional          # for the typed state dictionary

class AgentState(TypedDict):                      # shared state passed between LangGraph nodes
    # Input
    message      : str                             # the raw user message being analysed
    # Detection Agent outputs
    language     : Optional[str]                    # detected language code
    label        : Optional[str]                     # "scam" or "ham"
    confidence   : Optional[float]                    # classifier confidence for the predicted label
    scam_prob    : Optional[float]                     # probability assigned to the "scam" class
    # Explanation Agent outputs
    top_features : Optional[list]                      # top SHAP tokens
    rag_chunks   : Optional[list]                       # retrieved SPF advisory passages
    rag_summary  : Optional[str]                         # human-readable explanation summary
    # Risk Agent outputs
    scam_type    : Optional[str]                          # matched SPF taxonomy key
    spf_name     : Optional[str]                           # human-readable SPF scam name
    risk_score   : Optional[int]                            # 0-100 risk score
    risk_tier    : Optional[str]                             # CRITICAL / HIGH / MEDIUM / LOW / NONE
    advice       : Optional[str]                              # advisory text for the user
    # Final combined output
    output       : Optional[dict]                              # the final assembled result dict

print("AgentState schema defined")                # confirm schema is ready


AgentState schema defined


## Cell 7 — Define the Three Agent Nodes

Each agent is a **LangGraph node** — a function that receives the full shared state,
does its work, and returns updated state fields. This cell also defines the routing
function that decides whether to run the full analysis or skip to a safe exit, and
the safe-exit node itself.

- **Detection Agent**: XLM-RoBERTa classification + language detection
- **Routing function**: skips straight to a safe exit for clearly legitimate messages
- **Explanation Agent**: SHAP token attribution + **RAG retrieval from FAISS** (SPF corpus)
- **Risk Agent**: SPF taxonomy matching + risk score + advice
- **Safe-exit node**: builds a "no risk" result for messages the router skipped ahead on


In [6]:
# Each agent node is now a thin LangGraph wrapper around the shared
# classify() / explain() / get_risk() functions from scamsense_pipeline —
# no pipeline logic is duplicated here.

# Agent 1: Detection Agent
def detection_agent(state: AgentState) -> AgentState:        # LangGraph node: scam/ham detection
    """Classifies scam/ham + detects language via scamsense_pipeline.classify()."""
    det = classify(state["message"])                           # run the shared classifier
    return {
        **state,                                                # keep all existing state fields
        "language"  : det["language"],                          # store detected language
        "label"     : det["label"],                             # store predicted label
        "confidence": det["confidence"],                        # store prediction confidence
        "scam_prob" : det["scam_prob"],                         # store scam-class probability
    }


# Conditional routing after Detection Agent
def route_after_detection(state: AgentState) -> str:          # decides which node runs next
    """Skip Explanation + Risk agents if message is clearly safe (ham >= 80% confidence)."""
    if state["label"] == "ham" and state["confidence"] >= 0.80:  # confidently safe message?
        return "safe_exit"                                       # skip straight to safe exit
    return "explanation_agent"                                    # otherwise continue full analysis


# Agent 2: Explanation Agent (SHAP + RAG, via scamsense_pipeline.explain())
def explanation_agent(state: AgentState) -> AgentState:        # LangGraph node: SHAP + RAG explanation
    """Populates: top_features, rag_chunks, rag_summary (and a scam_type guess)."""
    exp = explain(state["message"], top_n=5, top_k=3)            # run the shared explanation step
    return {
        **state,                                                  # keep all existing state fields
        "top_features": exp["top_features"],                      # top SHAP tokens
        "rag_chunks"  : exp["rag_chunks"],                         # retrieved SPF passages
        "rag_summary" : exp["rag_summary"],                        # summary explanation text
        "_scam_type_guess": exp["scam_type"],                      # carried through to Risk Agent
    }


# Agent 3: Risk Agent (via scamsense_pipeline.get_risk())
def risk_agent(state: AgentState) -> AgentState:                 # LangGraph node: risk scoring
    """Maps the message to the SPF taxonomy, computes a risk score, selects advice."""
    scam_type = state.get("_scam_type_guess") or classify_scam_type(state["message"])[0]  # use guess, or re-derive it
    risk = get_risk(scam_type, state["scam_prob"])                 # compute risk score/tier/advice

    output = {                                                      # assemble the final result dict
        "message"         : state["message"],                       # original message
        "language"        : state["language"],                      # detected language
        "label"           : state["label"],                         # scam/ham label
        "confidence"      : state["confidence"],                    # classifier confidence
        "scam_prob"       : state["scam_prob"],                     # scam-class probability
        "top_features"    : state.get("top_features", []),          # SHAP tokens (empty if skipped)
        "rag_chunks"      : state.get("rag_chunks", []),             # RAG passages (empty if skipped)
        "rag_summary"     : state.get("rag_summary", ""),            # explanation text (empty if skipped)
        **risk,                                                       # merge in scam_type/risk_score/etc.
    }
    return {**state, **risk, "output": output}                       # update state and attach final output


# Safe exit for clearly legitimate messages
def safe_exit_node(state: AgentState) -> AgentState:                 # LangGraph node: early exit for safe messages
    output = {                                                        # build a "no risk" result dict
        "message"         : state["message"],                         # original message
        "language"        : state["language"],                        # detected language
        "label"           : "ham",                                     # force label to ham
        "confidence"      : state["confidence"],                       # classifier confidence
        "scam_prob"       : state["scam_prob"],                        # scam-class probability
        "top_features"    : [],                                        # no explanation needed
        "rag_chunks"      : [],                                        # no RAG lookup needed
        "rag_summary"     : "",                                        # no explanation text
        "scam_type"       : None,                                       # no scam type
        "spf_name"        : None,                                       # no SPF name
        "spf_cases_2025"  : None,                                        # no case stats
        "spf_losses_2025" : None,                                        # no loss stats
        "avg_loss_sgd"    : None,                                        # no average loss
        "risk_score"      : 0,                                           # zero risk
        "risk_tier"       : "NONE",                                      # no risk tier
        "advice"          : "This message appears legitimate. No action needed.",  # reassuring message
    }
    return {**state, "output": output}                                  # update state with the safe-exit output

print("All 3 agents + safe exit node defined (delegating to scamsense_pipeline)")  # confirm setup


All 3 agents + safe exit node defined (delegating to scamsense_pipeline)


## Cell 8 — Assemble LangGraph State Machine

In [7]:
from langgraph.graph import StateGraph, END       # LangGraph state machine + terminal marker

graph = StateGraph(AgentState)                     # create a new graph using our state schema

# Register all nodes
graph.add_node("detection_agent",   detection_agent)    # add detection node
graph.add_node("explanation_agent", explanation_agent)  # add explanation node
graph.add_node("risk_agent",        risk_agent)         # add risk-scoring node
graph.add_node("safe_exit",         safe_exit_node)     # add safe-exit node

# Entry point
graph.set_entry_point("detection_agent")           # every run starts at detection

# Conditional routing: ham with high confidence → skip to safe_exit
graph.add_conditional_edges(
    "detection_agent",                              # after detection...
    route_after_detection,                          # ...use this function to pick the next node
    {
        "safe_exit"        : "safe_exit",            # go to safe_exit if the function returns "safe_exit"
        "explanation_agent": "explanation_agent",    # otherwise go to explanation_agent
    }
)

# Scam path: Explanation → Risk → END
graph.add_edge("explanation_agent", "risk_agent")  # explanation always leads to risk scoring
graph.add_edge("risk_agent",        END)           # risk scoring ends the graph
graph.add_edge("safe_exit",         END)           # safe exit also ends the graph

pipeline = graph.compile()                         # compile the graph into a runnable pipeline

print("LangGraph pipeline compiled")               # confirm compilation
print("Nodes:", list(pipeline.get_graph().nodes.keys()))  # list all registered nodes


LangGraph pipeline compiled
Nodes: ['__start__', 'detection_agent', 'explanation_agent', 'risk_agent', 'safe_exit', '__end__']


## Cell 9 — Inference: Ten Test Cases (Five Scam + Five Legitimate, One per Language)


In [8]:
import pandas as pd  # dataframes for display tables
from IPython.display import display  # notebook-friendly table rendering

TIER_ICONS = {  # emoji icon per risk tier
    "CRITICAL": "🔴",
    "HIGH": "🟠",
    "MEDIUM": "🟡",
    "LOW": "🟢",
    "NONE": "✅"
}

LANG_NAMES = {  # language-code to display-name mapping
    "en": "English",
    "zh": "Mandarin",
    "ta": "Tamil",
    "ms": "Malay",
    "singlish": "Singlish"
}


def _show(df):  # helper: display a dataframe without the index column
    display(df.style.hide(axis="index"))

# Pipeline runner
def run_pipeline_display(name: str, message: str):  # runs the graph on one message and prints a report

    initial: AgentState = {  # build the starting state for this message
        "message": message,  # the input text
        "language": None,  # filled in by detection_agent
        "label": None,  # filled in by detection_agent
        "confidence": None,  # filled in by detection_agent
        "scam_prob": None,  # filled in by detection_agent
        "top_features": None,  # filled in by explanation_agent
        "rag_chunks": None,  # filled in by explanation_agent
        "rag_summary": None,  # filled in by explanation_agent
        "scam_type": None,  # filled in by risk_agent
        "spf_name": None,  # filled in by risk_agent
        "risk_score": None,  # filled in by risk_agent
        "risk_tier": None,  # filled in by risk_agent
        "advice": None,  # filled in by risk_agent
        "output": None,  # filled in by risk_agent / safe_exit_node
    }

    final = pipeline.invoke(initial)  # run the full LangGraph pipeline
    result = final["output"]  # extract the assembled result dict

    lang = LANG_NAMES.get(result["language"], result["language"])  # convert language code to display name
    tier = result.get("risk_tier", "NONE")  # get risk tier (default NONE)
    icon = TIER_ICONS.get(tier, "❓")  # matching icon (default "?")

    print(f"TEST CASE: {name.upper()}")  # section header
    print("-" * 90)  # divider line
    print(f"INPUT: {message}")  # show the original message

    # SUMMARY
    print("\nSUMMARY")  # section header

    if result["label"] == "scam":  # build a detailed table for scam predictions

        summary = pd.DataFrame([{
            "Test Case": name.upper(),                       # test case label
            "Prediction": result["label"].upper(),            # SCAM
            "Confidence": f"{result['confidence']:.1%}",      # classifier confidence
            "Language": lang,                                 # display language
            "Scam Prob": f"{result['scam_prob']:.1%}",        # scam-class probability
            "Risk Tier": f"{icon} {tier}",                    # risk tier with icon
            "Risk Score": f"{result['risk_score']}/100",      # numeric risk score
            "Scam Type": result["spf_name"],                  # SPF scam name
            "Tracked": "✅" if result.get("officially_tracked") else "—",  # (unused key; always shows "—")
            "SPF Cases": (  # 2025 case count, if tracked
                f"{result['spf_cases_2025']:,}"
                if result.get("spf_cases_2025") is not None else "N/A"
            ),
            "Total Losses": (  # 2025 losses, if tracked
                f"${result['spf_losses_2025']}M"
                if result.get("spf_losses_2025") is not None else "N/A"
            ),
            "Avg Loss": (  # average loss, if tracked
                f"${result['avg_loss_sgd']:,}"
                if result.get("avg_loss_sgd") is not None else "N/A"
            )
        }])

    else:                                                     # simpler table for ham predictions

        summary = pd.DataFrame([{
            "Test Case": name.upper(),                          # test case label
            "Prediction": result["label"].upper(),               # HAM
            "Confidence": f"{result['confidence']:.1%}",         # classifier confidence
            "Language": lang,                                    # display language
            "Scam Prob": f"{result['scam_prob']:.1%}",           # scam-class probability
        }])

    _show(summary)  # render the summary table

    # SHAP
    if result["label"] == "scam" and result.get("top_features"):  # only show SHAP table for scams with features

        print("\nTOP SHAP TOKENS")  # section header

        shap_tbl = pd.DataFrame([
            {
                "Rank": i + 1,  # rank order
                "Token": f["token"],  # the token text
                "SHAP Score": f"{f['shap_value']:+.4f}"  # signed SHAP contribution
            }
            for i, f in enumerate(result["top_features"])
        ])

        _show(shap_tbl)  # render the SHAP table

    # RAG
    if result["label"] == "scam" and result.get("rag_chunks"):  # only show RAG table for scams with chunks

        print("\nRAG RETRIEVAL")  # section header

        rag_tbl = pd.DataFrame([
            {
                "ID": c["id"],  # passage id
                "Type": c["scam_type"],  # passage's scam type
                "Score": f"{c['score']:.3f}",  # similarity score
                "Page": c["source_page"],  # source page number
                "Topic": c["topic"]  # passage topic
            }
            for c in result["rag_chunks"]
        ])

        _show(rag_tbl)  # render the RAG table

    if result["label"] == "scam":                            # print explanation text for scams only

        print("\nEXPLANATION")                                 # section header
        print(result["rag_summary"])                           # SHAP + RAG grounded summary

    print("\nADVICE")  # section header (always shown)
    print(result["advice"])  # what the user should do

    return result  # return the full result dict

# Test cases
test_cases = [  # one message per language: 5 scam types + 5 legitimate (ham) messages
    ("en / phishing",  # English phishing example
     "URGENT: Your POSB account has been locked due to suspicious activity. Verify immediately at https://posb-secure-login.xyz using your OTP or your account will be suspended."),

    ("ta / investment",  # Tamil investment-scam example
     "300% லாபம் உறுதி! இன்று முதலீடு செய்து 30 நாட்களில் உங்கள் பணத்தை மூன்று மடங்காக பெறுங்கள். இப்போதே Telegram-ல் பதிவு செய்யுங்கள்."),

    ("ms / job_scam",  # Malay job-scam example
     "Kerja dari rumah! Pendapatan RM8,000 sebulan tanpa pengalaman. Bayar yuran pendaftaran RM300 sekarang untuk mula bekerja."),

    ("singlish / fake_friend",  # Singlish fake-friend-scam example
     "Eh bro, this my new number lah. Wallet kena stolen overseas. Can PayNow me $500 first? Very urgent, I pay you back tomorrow confirm."),

    ("zh / government_impersonation",  # Mandarin government-impersonation example
     "您好，我是新加坡警察局警官。您的银行账户涉嫌洗钱，请立即将所有资金转入安全账户，否则今天会发出逮捕令。"),
    
    ("en / ham",  # English legitimate message
     "Hi John, your dental appointment is confirmed for tomorrow at 2:00 PM. Please arrive 10 minutes early."),

    ("ta / ham",  # Tamil legitimate message
     "வணக்கம்! நாளை காலை 10 மணிக்கு பள்ளியில் பெற்றோர்-ஆசிரியர் சந்திப்பு நடைபெறும்."),

    ("ms / ham",  # Malay legitimate message
     "Hai! Mesyuarat projek akan bermula pada pukul 3 petang esok di Bilik Mesyuarat A. Sila hadir tepat pada waktunya."),

    ("singlish / ham",  # Singlish legitimate message
     "Eh bro, we meeting at VivoCity 7pm tonight lah. Don't forget to bring your laptop for the project discussion."),

    ("zh / ham",  # Mandarin legitimate message
     "您好，提醒您明天下午2点在莱佛士医疗中心有预约。请提前10分钟到达，谢谢。"),
]

# Run
print("SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results")  # banner

all_results = []  # collect results for the summary table

for name, msg in test_cases:  # run every test case through the pipeline
    r = run_pipeline_display(name, msg)  # run pipeline + print report
    r["test_case"] = name  # tag result with its test case name
    all_results.append(r)  # store for the summary table later

print("\nAll test cases complete.")  # confirm completion

SCAMSENSE — LangGraph 3-Agent Pipeline: Inference Results
TEST CASE: EN / PHISHING
------------------------------------------------------------------------------------------
INPUT: URGENT: Your POSB account has been locked due to suspicious activity. Verify immediately at https://posb-secure-login.xyz using your OTP or your account will be suspended.

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob,Risk Tier,Risk Score,Scam Type,Tracked,SPF Cases,Total Losses,Avg Loss
EN / PHISHING,SCAM,100.0%,English,100.0%,🟠 HIGH,77/100,Phishing Scam,—,"6,264",$39.9M,"$6,384"



TOP SHAP TOKENS


Rank,Token,SHAP Score
1,lock,+0.0550
2,TP,+0.0401
3,cure,+0.0380
4,suspend,+0.0371
5,POS,+0.0363



RAG RETRIEVAL


...


ADVICE
This message appears legitimate. No action needed.
TEST CASE: SINGLISH / HAM
------------------------------------------------------------------------------------------
INPUT: Eh bro, we meeting at VivoCity 7pm tonight lah. Don't forget to bring your laptop for the project discussion.

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob
SINGLISH / HAM,HAM,100.0%,English,0.0%



ADVICE
This message appears legitimate. No action needed.
TEST CASE: ZH / HAM
------------------------------------------------------------------------------------------
INPUT: 您好，提醒您明天下午2点在莱佛士医疗中心有预约。请提前10分钟到达，谢谢。

SUMMARY


Test Case,Prediction,Confidence,Language,Scam Prob
ZH / HAM,HAM,100.0%,Mandarin,0.0%



ADVICE
This message appears legitimate. No action needed.

All test cases complete.


## Cell 10 — Summary Table

In [9]:
import pandas as pd  # dataframes for tables
from IPython.display import display  # display dataframes in notebook

rows = []  # collect one row per test case
for r in all_results:  # loop over every pipeline result
    top_tok = r["top_features"][0]["token"] if r.get("top_features") else "—"  # top SHAP token, if any
    rag_used = len(r.get("rag_chunks", []))  # number of RAG passages retrieved

    rows.append({
        "Test Case": r["test_case"],  # test case label
        "Prediction": r["label"].upper(),  # SCAM or HAM
        "Confidence": f"{r['confidence']:.1%}",  # classifier confidence
        "Language": LANG_NAMES.get(r["language"], r["language"]),  # display language name
        "Scam Type": r.get("spf_name") or "—",  # SPF scam name, if any
        "Risk Tier": f"{TIER_ICONS.get(r['risk_tier'], '')} {r['risk_tier']}",  # tier with icon
        "Risk Score": r["risk_score"],  # numeric risk score
        "Top SHAP Token": top_tok,  # most influential token
        "RAG Passages": rag_used,  # number of advisory passages used
    })

df_summary = pd.DataFrame(rows)  # build the full summary dataframe

print("  PIPELINE SUMMARY — ALL TEST CASES")  # banner

# Display as an interactive, scrollable table
display(df_summary)  # render the full table

# Save CSV
out_dir = BASE_DIR / "results"  # results output folder
out_dir.mkdir(parents=True, exist_ok=True)  # create it if missing

csv_path = out_dir / "06_agent_inference_results.csv"  # CSV output path
df_summary.to_csv(csv_path, index=False)  # save the summary table to disk

print(f"\nSummary table saved → {csv_path}")  # confirm save location

  PIPELINE SUMMARY — ALL TEST CASES


,Test Case,Prediction,Confidence,Language,Scam Type,Risk Tier,Risk Score,Top SHAP Token,RAG Passages
0,en / phishing,SCAM,100.0%,English,Phishing Scam,🟠 HIGH,77,lock,4
1,ta / investment,SCAM,100.0%,Tamil,Investment Scam,🔴 CRITICAL,95,Telegram,4
2,ms / job_scam,SCAM,100.0%,Malay,Job Scam,🟠 HIGH,80,bekerja,3
3,singlish / fake_friend,SCAM,99.7%,English,Fake Friend Call Scam,🟡 MEDIUM,61,urgent,3
4,zh / government_impersonation,SCAM,99.6%,Mandarin,Government Officials Impersonation Scam,🔴 CRITICAL,92,您,3
5,en / ham,HAM,100.0%,English,—,✅ NONE,0,—,0
6,ta / ham,HAM,100.0%,Tamil,—,✅ NONE,0,—,0
7,ms / ham,HAM,99.9%,Malay,—,✅ NONE,0,—,0
8,singlish / ham,HAM,100.0%,English,—,✅ NONE,0,—,0
9,zh / ham,HAM,100.0%,Mandarin,—,✅ NONE,0,—,0



Summary table saved → /kaggle/working/ScamSense/results/06_agent_inference_results.csv


## Cell 11 — Save Artifacts


In [10]:
import json  # for writing JSON files

# Save SPF taxonomy 
models_dir = BASE_DIR / "models"  # folder for saved model artifacts
models_dir.mkdir(parents=True, exist_ok=True)  # create it if missing

taxonomy_out = {  # strip out keyword lists before saving
    k: {ek: ev for ek, ev in v.items() if ek != "keywords"}
    for k, v in SPF_TAXONOMY.items()
}
with open(models_dir / "spf_taxonomy.json", "w") as f:  # write taxonomy to disk
    json.dump(taxonomy_out, f, indent=2)  # pretty-printed JSON
print("SPF taxonomy saved → models/spf_taxonomy.json")  # confirm save

# Save SPF corpus JSON for downstream notebooks
import shutil  # (imported but unused directly here)
with open(RAG_DIR / "spf_corpus.json", "w") as f:  # write the RAG corpus to disk
    json.dump(SPF_CORPUS, f, indent=2, ensure_ascii=False)  # preserve non-ASCII (Tamil/Chinese) text
print("SPF corpus saved   → rag/spf_corpus.json")  # confirm save
print("FAISS index already saved → rag/spf_faiss.index")  # index was cached earlier by init()

print("\n" + "-" * 64)  # divider
print("  Agent 1 : Detection      (XLM-RoBERTa + language detection)")  # recap
print("  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)")  # recap
print("  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)")  # recap

SPF taxonomy saved → models/spf_taxonomy.json
SPF corpus saved   → rag/spf_corpus.json
FAISS index already saved → rag/spf_faiss.index

----------------------------------------------------------------
  Agent 1 : Detection      (XLM-RoBERTa + language detection)
  Agent 2 : Explanation    (SHAP + FAISS RAG over SPF corpus)
  Agent 3 : Risk Scoring   (SPF 2025 taxonomy + advisory)
